# Colab runtime smoke test

Open this file in VS Code, click **Select Kernel** (top right) -> **Colab** -> pick an **L4** runtime,
then run the cells below.

Purpose: prove the assistant can drive the Colab GPU through the VS Code kernel. If it can,
the WSL2 + `google-colab-cli` route is unnecessary for this project.

See `ai-collab/handover-vlm-parser.md` for where this fits (stage P4a).

In [3]:
import os
import platform
import sys

print("python  ", sys.version.split()[0])
print("platform", platform.platform())
print("cwd     ", os.getcwd())
print("colab   ", os.path.exists("/content"))

python   3.12.13
platform Linux-6.6.122+-x86_64-with-glibc2.35
cwd      /content
colab    True


In [4]:
print("hi")

hi


In [5]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
NVIDIA L4, 23034 MiB, 580.82.07


In [6]:
# Which runtime did we actually get, and can it train in bf16?
#
# WARNING: `torch.cuda.is_bf16_supported()` defaults to `including_emulation=True`
# and returns True even on a Turing T4 (sm_75), which has no bf16 hardware at all.
# Measured on Colab 2026-08-22: emulation True, native False. Always ask for the
# native answer -- the emulated path runs, it is just not accelerated.
import torch

print("torch          ", torch.__version__)
print("cuda available ", torch.cuda.is_available())
if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability(0)
    print("device         ", torch.cuda.get_device_name(0))
    print("capability     ", capability)
    print("VRAM GiB       ", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    print("bf16 emulated  ", torch.cuda.is_bf16_supported(including_emulation=True))
    print("bf16 NATIVE    ", torch.cuda.is_bf16_supported(including_emulation=False))
    print("Ampere+        ", capability >= (8, 0))
    print()
    if capability >= (8, 0):
        print("-> bf16 LoRA is available: train Qwen3.5-4B.")
    else:
        print("-> No native bf16. Either pay for an L4/A100, or train Gemma 4 E4B with QLoRA.")

torch           2.11.0+cu128
cuda available  True
device          NVIDIA L4
capability      (8, 9)
VRAM GiB        22.03
bf16 emulated   True
bf16 NATIVE     True
Ampere+         True

-> bf16 LoRA is available: train Qwen3.5-4B.


In [7]:
import os, platform, sys
print("python  ", sys.version.split()[0])
print("platform", platform.platform())
print("cwd     ", os.getcwd())
print("colab   ", os.path.exists("/content"))
print("release ", os.environ.get("COLAB_RELEASE_TAG"))

python   3.12.13
platform Linux-6.6.122+-x86_64-with-glibc2.35
cwd      /content
colab    True
release  release-colab-external-images_20260715-060047_RC00


In [8]:
import subprocess
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
     "--format=csv"], capture_output=True, text=True).stdout)

import torch
print("torch          ", torch.__version__)
print("cuda available ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device         ", torch.cuda.get_device_name(0))
    print("capability     ", torch.cuda.get_device_capability(0))
    print("bf16 supported ", torch.cuda.is_bf16_supported())
    print("total VRAM GiB ", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

name, memory.total [MiB], driver_version, compute_cap
NVIDIA L4, 23034 MiB, 580.82.07, 8.9

torch           2.11.0+cu128
cuda available  True
device          NVIDIA L4
capability      (8, 9)
bf16 supported  True
total VRAM GiB  22.03


In [9]:
import inspect, torch
print("signature:", inspect.signature(torch.cuda.is_bf16_supported))
try:
    print("bf16 including emulation :", torch.cuda.is_bf16_supported(including_emulation=True))
    print("bf16 NATIVE hardware     :", torch.cuda.is_bf16_supported(including_emulation=False))
except TypeError as e:
    print("no including_emulation arg:", e)
print("cap >= (8,0) i.e. Ampere+ :", torch.cuda.get_device_capability(0) >= (8, 0))

signature: (including_emulation: bool = True)
bf16 including emulation : True
bf16 NATIVE hardware     : True
cap >= (8,0) i.e. Ampere+ : True


In [10]:
# How much does the lack of native bf16 actually cost? Compare against fp16,
# which Turing *does* accelerate.
import time

import torch


def bench(dtype, n=4096, iters=30):
    a = torch.randn(n, n, device="cuda", dtype=dtype)
    b = torch.randn(n, n, device="cuda", dtype=dtype)
    for _ in range(5):
        a @ b
    torch.cuda.synchronize()
    started = time.perf_counter()
    for _ in range(iters):
        a @ b
    torch.cuda.synchronize()
    seconds = (time.perf_counter() - started) / iters
    return seconds * 1000, 2 * n**3 / seconds / 1e12


for dtype in (torch.float32, torch.float16, torch.bfloat16):
    milliseconds, tflops = bench(dtype)
    print(f"{str(dtype):16s} {milliseconds:7.2f} ms   {tflops:6.2f} TFLOP/s")

torch.float32      11.05 ms    12.43 TFLOP/s
torch.float16       2.26 ms    60.74 TFLOP/s
torch.bfloat16      2.14 ms    64.11 TFLOP/s


In [11]:
import os, subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("capability  ", torch.cuda.get_device_capability(0))
print("bf16 NATIVE ", torch.cuda.is_bf16_supported(including_emulation=False))

NVIDIA L4, 23034 MiB, 8.9
capability   (8, 9)
bf16 NATIVE  True


In [12]:
print("hi")

hi
